## 2.5自动求导
求导是几乎所有深度学习优化算法的关键步骤。 虽然求导的计算很简单，只需要一些基本的微积分。 但对于复杂的模型，手工进行更新是一件很痛苦的事情（而且经常容易出错）。

深度学习框架通过自动计算导数，即自动微分（automatic differentiation）来加快求导。 实际中，根据设计好的模型，系统会构建一个计算图（computational graph）， 来跟踪计算是哪些数据通过哪些操作组合起来产生输出。 自动微分使系统能够随后反向传播梯度。 这里，反向传播（backpropagate）意味着跟踪整个计算图，填充关于每个参数的偏导数。

### 2.5.1一个简单的例子

作为一个演示例子，(**假设我们想对函数$y=2\mathbf{x}^{\top}\mathbf{x}$关于列向量$\mathbf{x}$求导**)。
首先，我们创建变量`x`并为其分配一个初始值。

In [3]:
import torch
x = torch.arange(4.0)
x


tensor([0., 1., 2., 3.])

[**在我们计算$y$关于$\mathbf{x}$的梯度之前，需要一个地方来存储梯度。**]
重要的是，我们不会在每次对一个参数求导时都分配新的内存。
因为我们经常会成千上万次地更新相同的参数，每次都分配新的内存可能很快就会将内存耗尽。
注意，一个标量函数关于向量$\mathbf{x}$的梯度是向量，并且与$\mathbf{x}$具有相同的形状。


In [4]:
x = torch.arange(4.0, requires_grad=True)
# requires_grad=True: 这是关键参数，表示PyTorch需要跟踪这个张量的所有操作，以便后续计算梯度。
x,x.grad 
# 访问张量x的梯度属性
# 如果x是某个函数的输入，并且我们计算了这个函数相对于x的梯度，那么梯度值会存储在x.grad中

(tensor([0., 1., 2., 3.], requires_grad=True), None)

(**现在计算$y$。**)


In [5]:
y = 2 * torch.dot(x,x)#内积
y

tensor(28., grad_fn=<MulBackward0>)

`x`是一个长度为4的向量，计算`x`和`x`的点积，得到了我们赋值给`y`的标量输出。
接下来，[**通过调用反向传播函数来自动计算`y`关于`x`每个分量的梯度**]，并打印这些梯度。


In [6]:
y.backward()
x.grad

tensor([ 0.,  4.,  8., 12.])

In [7]:
x.grad == 4*x

tensor([True, True, True, True])

[**现在计算`x`的另一个求和函数的梯度。**]

数学原理剖析（为什么梯度全是 1？）

这行代码背后的数学逻辑异常简单。我们的输入向量是 $\mathbf{x} = [x_1, x_2, x_3, x_4]^\top$。sum() 函数的数学表达式为：$$y = x_1 + x_2 + x_3 + x_4$$当我们执行 y.backward() 时，就是要求标量 $y$ 对向量 $\mathbf{x}$ 的梯度。根据我们之前讨论的，标量对向量求导，结果是一个与 $\mathbf{x}$ 形状相同的向量：$$\nabla_{\mathbf{x}} y = \left[ \frac{\partial y}{\partial x_1}, \frac{\partial y}{\partial x_2}, \frac{\partial y}{\partial x_3}, \frac{\partial y}{\partial x_4} \right]^\top$$现在，我们单独看每一个偏导数。比如对 $x_1$ 求偏导：$$\frac{\partial y}{\partial x_1} = \frac{\partial (x_1 + x_2 + x_3 + x_4)}{\partial x_1}$$因为在对 $x_1$ 求导时，$x_2, x_3, x_4$ 都被视为常数，常数的导数是 0，而 $x_1$ 自己的导数是 1。所以：$$\frac{\partial y}{\partial x_1} = 1 + 0 + 0 + 0 = 1$$同理可得，对 $x_2, x_3, x_4$ 的偏导数也全都是 1。因此，最终算出的梯度向量就是：$$\nabla_{\mathbf{x}} y = [1.0, 1.0, 1.0, 1.0]^\top$$如果你在代码最后打印 print(x.grad)，得到的结果绝对就是这个。

底层机制：无视输入的“线性传递者”

从这个简单的例子中，你能顿悟深度学习框架处理 sum() 操作的一个核心逻辑：加法节点（求和节点）在反向传播时，是一个完全公平的“梯度分发器”。无论输入的 $x_i$ 是 $0$ 还是 $10000$，偏导数 $\frac{\partial y}{\partial x_i}$ 永远是 $1$。它会把上一层传过来的梯度，原封不动、等额地复制分发给所有参与求和的输入变量。


In [10]:
# 在默认情况下，PyTorch会累积梯度。这意味着如果你多次调用backward()，梯度会被累加到现有的梯度上，而不是覆盖它们。
# 这就是为什么在每次调用backward()之前，我们通常需要将梯度清零，以避免梯度累积的影响。
x.grad.zero_()#将梯度清零
y = x.sum()
y.backward()
x,y,x.grad

(tensor([0., 1., 2., 3.], requires_grad=True),
 tensor(6., grad_fn=<SumBackward0>),
 tensor([1., 1., 1., 1.]))

### 2.5.2非标量变量的反向传播

当`y`不是标量时，向量`y`关于向量`x`的导数的最自然解释是一个矩阵。
对于高阶和高维的`y`和`x`，求导的结果可以是一个高阶张量。

然而，虽然这些更奇特的对象确实出现在高级机器学习中（包括[**深度学习中**]），
但当调用向量的反向计算时，我们通常会试图计算一批训练样本中每个组成部分的损失函数的导数。
这里(**，我们的目的不是计算微分矩阵，而是单独计算批量中每个样本的偏导数之和。**)


In [ ]:
# 对非标量调用backward()时，需要指定一个gradient参数，该参数指定关于标量输出的梯度。
# 本例只想求偏导数的和，因此传递一个1的梯度是合适的
x.grad.zero_()
y = x * x#y是一个向量
# 等价于 y.backward(torch.ones(len(x)))，因为我们想要每个元素的梯度都为1
y.sum().backward()
x,y,x.grad

(tensor([0., 1., 2., 3.], requires_grad=True),
 tensor([0., 1., 4., 9.], grad_fn=<MulBackward0>),
 tensor([0., 2., 4., 6.]))

这段代码看似简单，但它那句注释（`等价于 y.backward(torch.ones(len(x)))`）却道出了 PyTorch 处理非标量反向传播（Vector-Jacobian Product, VJP）的核心机密。

我们把这段代码分两步来彻底拆解：

#### 第一步：`y = x * x`（向量运算）

这里的 `*` 是**逐元素相乘（Element-wise）**，而不是矩阵乘法。
数学表达式为：


$$y = [x_1^2, x_2^2, x_3^2, x_4^2]^\top = [0^2, 1^2, 2^2, 3^2]^\top = [0, 1, 4, 9]^\top$$

此时，$\mathbf{y}$ 是一个**向量**，而不是标量。

---

#### 第二步：为什么必须加 `.sum()` 再 `.backward()`？

如果你在这里直接运行 `y.backward()`，PyTorch 会直接报错：*`grad can be implicitly created only for scalar outputs`*（只能为标量输出隐式创建梯度）。

为什么？因为我们在前几轮讨论过，向量对向量求导会产生一个庞大的**雅可比矩阵（Jacobian）**。PyTorch 极其抗拒在内存里去实例化一个完整的矩阵，它只想算**向量（梯度）乘以矩阵**。

为了避免报错，框架要求你要么：

1. **给出一个外部传进来的梯度向量**，比如 `y.backward(torch.ones(len(x)))`。
2. **把输出变成标量**，比如通过 `y.sum()`。

##### 为什么 `y.sum()` 完美等价于 `torch.ones`？

这就是深度学习里极其聪明的一个**数学小把戏**。
我们设 $L = y.\text{sum}()$。
展开来看：


$$L = y_1 + y_2 + y_3 + y_4$$

现在，我们要通过 $L$ 对 $x$ 求导。根据链式法则：


$$\frac{\partial L}{\partial x_i} = \frac{\partial L}{\partial y_i} \times \frac{\partial y_i}{\partial x_i}$$

1. **先看左半边 $\frac{\partial L}{\partial y_i}$**：
因为 $L$ 是简单相加，所以无论 $y_i$ 是多少，$\frac{\partial L}{\partial y_i}$ **永远等于 1**。
所以，传给中间层 $\mathbf{y}$ 的梯度向量就是 $[1, 1, 1, 1]^\top$。**这不就是 `torch.ones(len(x))` 吗！**
2. **再看右半边 $\frac{\partial y_i}{\partial x_i}$**：
因为 $y_i = x_i^2$，所以导数是 $2x_i$。
3. **两者相乘**：

$$\frac{\partial L}{\partial x_i} = 1 \times 2x_i = 2x_i$$



把所有的 $2x_i$ 拼起来，就得到了最终的梯度：


$$\nabla_{\mathbf{x}} L = [2(0), 2(1), 2(2), 2(3)]^\top = [0, 2, 4, 6]^\top$$

#### 💡 核心顿悟

在实际的深度学习写代码（比如计算机视觉或 NLP）中，我们的神经网络最终输出的通常是一个包含成千上万个元素的预测向量（比如每个像素的分类概率，或者每个框的坐标）。
我们计算出损失（Loss）向量后，**几乎总是会用 `.sum()` 或者 `.mean()` 把它变成一个标量**，然后再去调用 `.backward()`。

这个 `.sum()` 操作，本质上就是巧妙地把一个全为 1 的初始梯度向量 `[1, 1, ..., 1]`，温柔地注入到整个计算图的末端，从而开启反向传播的连锁反应。

### 2.5.3分离计算

有时，我们希望[**将某些计算移动到记录的计算图之外**]。
例如，假设`y`是作为`x`的函数计算的，而`z`则是作为`y`和`x`的函数计算的。
想象一下，我们想计算`z`关于`x`的梯度，但由于某种原因，希望将`y`视为一个常数，
并且只考虑到`x`在`y`被计算后发挥的作用。

这里可以分离`y`来返回一个新变量`u`，该变量与`y`具有相同的值，
但丢弃计算图中如何计算`y`的任何信息。
换句话说，梯度不会向后流经`u`到`x`。
因此，下面的反向传播函数计算`z=u*x`关于`x`的偏导数，同时将`u`作为常数处理，
而不是`z=x*x*x`关于`x`的偏导数。


In [ ]:
x.grad.zero_()
y = x * x
u = y.detach()#分离计算图
z = u * x

z.sum().backward()
x.grad == u#将u分离出去，视为常数，因此z对x的梯度就是u

tensor([True, True, True, True])

### 2.5.4Python控制流的梯度计算

使用自动微分的一个好处是：
[**即使构建函数的计算图需要通过Python控制流（例如，条件、循环或任意函数调用），我们仍然可以计算得到的变量的梯度**]。
在下面的代码中，`while`循环的迭代次数和`if`语句的结果都取决于输入`a`的值。


In [ ]:
def f(a):
    b = a * 2
    while b.norm() < 1000:#当b的2范数小于1000时，继续循环
        b = b * 2
    if b.sum() > 0:#如果b的元素之和大于0，则c等于b，否则c等于100*b
        c = b
    else:
        c = 100 * b
    return c

# 计算梯度
a = torch.randn(size=(), requires_grad=True)#size=()创建一个0维张量（标量）
d = f(a)#d是一个标量就是c
d.backward()
a,d,a.grad == d / a

(tensor(1.7753, requires_grad=True),
 tensor(1817.8740, grad_fn=<MulBackward0>),
 tensor(True))